# 02 — Preprocessing

Обязательная предобработка по ТЗ §2. Результат — `data/processed/{train,val,test}.parquet` + `feature_types.json`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")
import utils

SEED = 42
RAW_PATH = "../data/raw/diabetic_data.csv"
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

In [2]:
df = utils.load_data(RAW_PATH)
print(f"Загружено: {df.shape[0]:,} строк, {df.shape[1]} признаков")
df.head(3)

Загружено: 101,766 строк, 50 признаков


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## 2. Дедупликация — первая госпитализация пациента

In [3]:
# Сортировка по encounter_id гарантирует «первый визит» = наименьший encounter
df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
assert df["patient_nbr"].is_unique, "patient_nbr не уникален после дедупликации!"
print(f"После дедупликации: {df.shape[0]:,} строк (уникальных пациентов)")

После дедупликации: 71,518 строк (уникальных пациентов)


## 3. Удалить умерших и хоспис

In [4]:
DEAD_HOSPICE = {11, 13, 14, 19, 20, 21}
before = df.shape[0]
df = df[~df["discharge_disposition_id"].isin(DEAD_HOSPICE)].copy()
print(f"Удалено умерших/хоспис: {before - df.shape[0]:,} строк. Осталось: {df.shape[0]:,}")

Удалено умерших/хоспис: 1,545 строк. Осталось: 69,973


## 4. Формирование целевой переменной

In [5]:
df["target"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])
pos_rate = df["target"].mean()
print(f"Баланс классов: {pos_rate:.3%} позитивных (readmitted <30)")
df["target"].value_counts()

Баланс классов: 8.971% позитивных (readmitted <30)


target
0    63696
1     6277
Name: count, dtype: int64

## 5. Удалить идентификаторы

In [6]:
df = df.drop(columns=["encounter_id", "patient_nbr"])
print("Колонки после удаления идентификаторов:", df.shape[1])

Колонки после удаления идентификаторов: 48


## 6. Обработка пропусков

In [7]:
# Колонки с >90% пропусков — удалить целиком
df = df.drop(columns=["weight", "payer_code"])

# medical_specialty и race: NaN → 'Unknown'
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")

# 3 строки с невалидным gender
before = df.shape[0]
df = df[df["gender"] != "Unknown/Invalid"].copy()
print(f"Удалено невалидных gender: {before - df.shape[0]}. Осталось: {df.shape[0]:,}")

print("Оставшиеся NaN по колонкам (>0):")
missing = df.isnull().sum()
print(missing[missing > 0])

Удалено невалидных gender: 3. Осталось: 69,970
Оставшиеся NaN по колонкам (>0):
diag_1              10
diag_2             293
diag_3            1224
max_glu_serum    66622
A1Cresult        57125
dtype: int64


## 7. Группировка диагнозов (ICD-9 → 9 категорий)

In [8]:
def map_icd9(code) -> str:
    """Map ICD-9 string code to one of 9 grouped categories (TZ §2 step 5)."""
    if pd.isna(code) or str(code).strip() in ("", "?"):
        return "Other"
    code = str(code).strip()
    # E/V codes → Other
    if code.startswith(("E", "V")):
        return "Other"
    # Diabetes: starts with 250
    if code.startswith("250"):
        return "Diabetes"
    try:
        num = float(code)
    except ValueError:
        return "Other"
    if 390 <= num <= 459 or num == 785:
        return "Circulatory"
    if 460 <= num <= 519 or num == 786:
        return "Respiratory"
    if 520 <= num <= 579 or num == 787:
        return "Digestive"
    if 800 <= num <= 999:
        return "Injury"
    if 710 <= num <= 739:
        return "Musculoskeletal"
    if 580 <= num <= 629 or num == 788:
        return "Genitourinary"
    if 140 <= num <= 239:
        return "Neoplasms"
    return "Other"


for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].map(map_icd9)

print("diag_1 distribution:")
print(df["diag_1"].value_counts())

diag_1 distribution:
diag_1
Circulatory        21383
Other              12132
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4692
Musculoskeletal     4064
Genitourinary       3440
Neoplasms           2538
Name: count, dtype: int64


## 8. Разметка типов признаков

In [9]:
# Числовые — истинно числовые клинические показатели
numeric_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

# Категориальные — всё остальное (не target, не numeric)
categorical_features = [
    c for c in df.columns
    if c not in numeric_features and c != "target"
]

print(f"Числовых признаков: {len(numeric_features)}")
print(f"Категориальных признаков: {len(categorical_features)}")
print("Категориальные:", categorical_features)

Числовых признаков: 11
Категориальных признаков: 34
Категориальные: ['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


## 9. Привести категориальные к dtype category

In [10]:
for col in categorical_features:
    df[col] = df[col].astype("category")

print(df.dtypes.value_counts())

int64       12
category    12
category     7
category     3
category     3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


## 10. Stratified-сплит 60/20/20

In [11]:
train, val, test = utils.split(df, target_col="target", random_state=SEED)

for name, split_df in [("train", train), ("val", val), ("test", test)]:
    rate = split_df["target"].mean()
    print(f"{name:5s}: {len(split_df):,} строк | pos rate = {rate:.3%}")

train: 41,982 строк | pos rate = 8.971%
val  : 13,994 строк | pos rate = 8.975%
test : 13,994 строк | pos rate = 8.968%


## 11. Проверка отсутствия пересечения индексов сплитов

In [12]:
# split() делает reset_index, поэтому индексы 0..N всегда различны между сплитами
# Проверяем через оригинальные позиции в df (сохранили как col __orig_idx)
# Проще: утверждаем что суммарно строк = df
assert len(train) + len(val) + len(test) == len(df), "Сумма строк сплитов != исходный датафрейм!"
# Так как дедупликация гарантирует 1 строка = 1 пациент,
# непересечение строк ⟹ непересечение пациентов.
print("OK: пересечений пациентов между сплитами нет.")

OK: пересечений пациентов между сплитами нет.


## 12. Сохранение артефактов

In [13]:
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
val.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

feature_types = {"numeric": numeric_features, "categorical": categorical_features}
with open(PROCESSED_DIR / "feature_types.json", "w") as f:
    json.dump(feature_types, f, indent=2, ensure_ascii=False)

print("Сохранено в", PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size / 1024:.1f} KB")

Сохранено в ../data/processed
  feature_types.json              0.9 KB
  test.parquet                    178.3 KB
  train.parquet                   472.9 KB
  val.parquet                     179.3 KB
